# 05: DPO, strict preference data, and RSFT

[Course index](../README_en.md) | [中文版本](../notebooks/05_alignment.ipynb)

**Goal:** derive DPO and understand why preference-pair quality matters more than a decreasing training loss.


## 1. From KL-regularized RL to DPO

$$\max_\pi\;\mathbb E_\pi[r(x,y)]-eta D_{KL}(\pi\Vert\pi_{ref}).$$

The optimum satisfies $\pi^*(y|x)\propto\pi_{ref}(y|x)e^{r/eta}$. Substituting the implied reward into a Bradley–Terry preference model yields

$$\mathcal L_{DPO}=-\log\sigma\left(eta[(\log\pi_	heta(y_w)-\log\pi_	heta(y_l))-(\log\pi_{ref}(y_w)-\log\pi_{ref}(y_l))]ight).$$


### Hands-on check

Load the production DPO components.


In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from tokenizer.tokenizer_utils import MiniLLMTokenizer
from train.dpo import DPODataset, dpo_loss
tok = MiniLLMTokenizer(str(ROOT / 'tokenizer/minillm_tokenizer.json'))


### Hands-on check

Verify that improving chosen relative to rejected lowers DPO loss.


In [ ]:
ref_c = torch.tensor([-2.0])
ref_r = torch.tensor([-2.0])
policy_c = torch.tensor([-1.5])
policy_r = torch.tensor([-2.5])
good_loss = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=1.0)
neutral_loss = dpo_loss(ref_c, ref_r, ref_c, ref_r, beta=1.0)
print('improved preference loss:', good_loss.item())
print('neutral loss (log 2):   ', neutral_loss.item())
assert good_loss < neutral_loss


### Hands-on check

Sweep the preference margin and observe monotonic loss.


In [ ]:
for margin in [-2, -1, 0, 1, 2]:
    loss = dpo_loss(torch.tensor([float(margin)]), torch.tensor([0.]), torch.tensor([0.]), torch.tensor([0.]), beta=1.)
    print(f'margin={margin:+d}, loss={loss.item():.4f}')


## 2. Strict preference data

If chosen/rejected differ mainly in length or formatting, DPO learns shortcuts. DPO-v2 keeps only QA, sentence-count, and keyword pairs where chosen hard-passes and rejected hard-fails, balances tasks, and excludes free continuation.

```bash
python scripts/build_strict_dpo.py
python train/dpo.py --config configs/train_instruction_dpo_v2.yaml
```


### Hands-on check

Load a strict DPO-v2 pair and inspect its supervised response region.


In [ ]:
ds = DPODataset(str(ROOT / 'data/instruction_dpo_v2/train.jsonl'), tok, max_length=256)
chosen_ids, chosen_labels, _, _ = ds[0]
valid = chosen_labels != -100
print('sequence tokens:', len(chosen_ids))
print('supervised tokens:', int(valid.sum()))
print('first supervised target:', tok.decode([int(chosen_labels[valid][0])]))
assert valid.any() and (~valid).any()


## 3. RSFT

Generate $K$ candidates, score them, select

$$y^*=rg\max_{y\in\{y_1,\ldots,y_K\}}R(x,y),$$

then run response-only SFT. RSFT is cheaper than online RL but inherits candidate and reward biases.


### Hands-on check

Select the hard-passing candidate with maximum reward.


In [ ]:
candidates = [
    {"text": "bird bird bird", "reward": 0.10, "hard_pass": False},
    {"text": "The blue bird flew home.", "reward": 0.85, "hard_pass": True},
    {"text": "A bird slept.", "reward": 0.45, "hard_pass": False},
]
eligible = [item for item in candidates if item["hard_pass"]]
selected = max(eligible, key=lambda item: item["reward"])
print(selected)
assert selected["text"] == "The blue bird flew home."


## 4. Reward overoptimization

A model can raise preference accuracy by suppressing rejected answers while PPL and task success degrade. Always compare preference diagnostics with external task metrics and language modeling.

- [Previous](04_sft.ipynb) · [Index](../README_en.md) · [Next](06_evaluation.ipynb)
- [`train/dpo.py`](../../../train/dpo.py) · [`scripts/build_strict_dpo.py`](../../../scripts/build_strict_dpo.py)


### Hands-on check

Use formal project numbers to show proxy improvement and external regression together.


In [ ]:
sft = {"preference_accuracy": 0.5833, "ppl": 5.80, "qa_em": 0.85}
dpo_v1 = {"preference_accuracy": 0.9167, "ppl": 6.34, "qa_em": 0.80}
print("preference gain:", dpo_v1["preference_accuracy"] - sft["preference_accuracy"])
print("PPL regression:", dpo_v1["ppl"] - sft["ppl"])
print("QA change:", dpo_v1["qa_em"] - sft["qa_em"])
assert dpo_v1["preference_accuracy"] > sft["preference_accuracy"]
assert dpo_v1["ppl"] > sft["ppl"] and dpo_v1["qa_em"] < sft["qa_em"]
